In [8]:
import { vectorSearch, rewriteQuery, chunksToMarkdown } from './src/lib/rag.mjs';
import fs from 'node:fs';
import path from 'node:path';
import { ChatOpenAI } from '@langchain/openai';
import { OPENAI_API_KEY } from './src/lib/vars.mjs';

function logFile(logContent, fileName = 'jupyter.md') {
  const logFileName = `logs/${fileName}`;
  const logDir = path.dirname(logFileName);
  if (!fs.existsSync(logDir)) {
    fs.mkdirSync(logDir, { recursive: true });
  }
  fs.writeFileSync(logFileName, '```markdown\n' + logContent + '\n```', 'utf8');
  console.log(logContent);
}

const gpt5 = new ChatOpenAI({
  modelName: 'gpt-5',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt41 = new ChatOpenAI({
  modelName: 'gpt-4.1',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});
const gpt5Mini = new ChatOpenAI({
  modelName: 'gpt-5-mini',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt4oMini = new ChatOpenAI({
  modelName: 'gpt-4o-mini',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});
const gpt5Nano = new ChatOpenAI({
  modelName: 'gpt-5-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt4oNano = new ChatOpenAI({
  modelName: 'gpt-4o-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});


In [2]:
const redditQuestion = `American with passport card, entered Canada a week ago for a concert, told the border officer he will return tomorrow, entry went smooth and easy
But now he wants to stay for another week to go to another event he could get tickets for. He has sufficient funds and will leave next week
Does he need to leave? Or could he stay an extra week despite saying he would return tomorrow? Should he call the border or someone saying his plans changed? He doesn’t have a Reddit account so I’m asking on his behalf`

async function pickQuery(question, model) {
  const { answer: query } = await rewriteQuery(question);
  console.log(`${model.model}: ${query}\n`);
  return query;
}

const [query] = await Promise.all([
  pickQuery(redditQuestion, gpt5Nano),
  pickQuery(redditQuestion, gpt4oNano),
  pickQuery(redditQuestion, gpt4oMini),
  pickQuery(redditQuestion, gpt5Mini),
])

gpt-4o-nano: What are the legal requirements and procedures for a U.S. citizen with a passport card who entered Canada for a temporary visit, initially intending to leave the next day, but now wishes to extend their stay by one week for additional events, and do they need to notify Canadian border authorities or immigration officials about this change in plans?

gpt-5-nano: What are the legal requirements and procedures for a U.S. citizen with a passport card who entered Canada for a temporary visit, initially intending to stay for one day, but now wishes to extend their stay by an additional week for attending another event, considering they have sufficient funds and plan to leave Canada after the extended period? Should they contact Canadian immigration authorities or border officials to notify them of the change in plans?

gpt-5-mini: What are the legal requirements and procedures for a U.S. citizen with a passport card who entered Canada for a temporary visit, initially intending t

In [9]:
const chunks = await vectorSearch(query);
logFile(chunksToMarkdown(chunks), 'chunks.md');

# Guide 5551 - Applying to Change Conditions or Extend Your Stay in Canada - online application

[Print](javascript:window.print\(\);)

## You’re seeing the instructions to apply online.

[Guide - apply on paper](/en/immigration-refugees-citizenship/services/application/application-forms-guides/guide-5551-applying-change-conditions-extend-your-stay-canada-paper.html)

Most people must apply online. If you can’t apply online because of a disability or problem with the online application, you can apply on paper.

## Table of contents

*   [Overview](#overview)
*   [Status in Canada](#5551E2)
*   [Restoration of status](#restoration)
*   [Completing the forms](#5551E4)
*   [Paying the fees](#5551E5)
*   [Submitting your application](#5551E6)
*   [What happens next?](#whatnext)

### You’re seeing the instructions to apply online.

Most people **must** apply online.

\[...\]

* * * 

 #### 2\. Temporary residents travelling without passports:

If you did not require a passport to enter Cana

In [ ]:
import * as z from 'zod';

async function pickChunks(model) {
  const { content: selectedChunks } = await model.invoke(
    `I'll give you a markdown content with a list of results from a vector search for a question.
  Remove the chunks that do not help to answer th question. Do not alter anything else.
  Pay attention to the header of the chunk to recognize if it's related to the topic we want to answer.
  Return the updated markdown as your answer (do not add any initial backquotes).

  **Question:** ${query}

  **Chunks:**

  \`\`\`markdown
  ${chunksToMarkdown(chunks)}
  \`\`\`
  `
  );

  logFile(selectedChunks, `picked-chunks-${model.model}.md`);
  return selectedChunks
}

const pickedChunks = await Promise.all([
  pickChunks(gpt5Nano),
  pickChunks(gpt41),
  pickChunks(gpt5Mini),
  pickChunks(gpt4oMini),
])
  .then(() => console.log('done'))
  .catch(console.error)

# Guide 5551 - Applying to Change Conditions or Extend Your Stay in Canada - online application

[Print](javascript:window.print\(\);)

## You’re seeing the instructions to apply online.

[Guide - apply on paper](/en/immigration-refugees-citizenship/services/application/application-forms-guides/guide-5551-applying-change-conditions-extend-your-stay-canada-paper.html)

Most people must apply online. If you can’t apply online because of a disability or problem with the online application, you can apply on paper.

## Table of contents

*   [Overview](#overview)
*   [Status in Canada](#5551E2)
*   [Restoration of status](#restoration)
*   [Completing the forms](#5551E4)
*   [Paying the fees](#5551E5)
*   [Submitting your application](#5551E6)
*   [What happens next?](#whatnext)

### You’re seeing the instructions to apply online.

Most people **must** apply online.

\[...\]

* * * 

 #### 2\. Temporary residents travelling without passports:

If you did not require a passport to enter Cana

**Conclusions:**

| Model       | Response Time | Performance | Notes                                           |
| ----------- | ------------- | ----------- | ----------------------------------------------- |
| GPT-4o-mini | 0:20          | Good        |                                                 |
| GPT-5       | 3:00+         | Best        | The only one to remove archived documents.      |
| GPT-5-mini  | 1:10          | Good        |                                                 |
| GPT-4.1     | 3:30          | Good        |                                                 |
| GPT-5-nano  | 0:30          | Mediocre    | Included unrelated documents (eg: work permit). |
